# Custom CNN
Experiment log: train (everything goes to MLflow) → submit → next experiment → submit.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")

import pandas as pd
import torch
from torchmetrics.functional.classification import multilabel_f1_score

from src.classifier import predict
from src.cnn import train
from src.config import DATA_DIR, FOLDS_PATH, LABELS, ROOT
from src.postprocess import add_parents, best_thresholds, blend_duplicates, hflip, phash

folds = pd.read_csv(FOLDS_PATH).set_index("image_id")


def val_f1(pred: pd.DataFrame, y: pd.DataFrame) -> float:
    return multilabel_f1_score(torch.tensor(pred.to_numpy(dtype="float32")), torch.tensor(y.to_numpy()), len(LABELS)).item()

## Experiment 1: baseline CNN

In [2]:
trainer = train()
dm = trainer.datamodule
RUN_ID = trainer.logger.run_id

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model     | CNN               | 2.8 M  | train | 0    
1 | criterion | BCEWithLogitsLoss | 0      | train | 0    
2 | train_f1  | MultilabelF1Score | 0      | train | 0    
3 | val_f1    | MultilabelF1Score | 0      | train | 0    
----------------------------------------------------------------
2.8 M     Trainable params
0         Non-trainable params
2.8 M     Total params
11.185    Total estimated model params size (MB)
95        Modules in train mode
0         Modules in eval mode
0         Total Flops
/Users/zaharguzij/Documents/GitHub/

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


MLflow experiment: dl_lab-2; run: 86ea605c7c8b46d98aa72fc2bfd99e25
Best checkpoint: /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/86ea605c7c8b46d98aa72fc2bfd99e25/cnn-epoch=09-val_f1=0.3312.ckpt


In [3]:
val = predict(trainer, dataloaders=dm.val_dataloader())
test = predict(trainer, datamodule=dm)
y = folds.loc[val.index, LABELS]
val_f1(val, y)

/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

0.33115750551223755

In [4]:
path = ROOT / "models" / RUN_ID / "submission.csv"
(test >= 0.5).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} baseline"

100%|█████████████████████████████████████████| 930k/930k [00:01<00:00, 519kB/s]
Successfully submitted to Cats and Dogs++

In [5]:
!kaggle competitions submissions -c cats-and-dogs | head -4

fileName        date                        description                                status                    publicScore  privateScore  
--------------  --------------------------  -----------------------------------------  ------------------------  -----------  ------------  
submission.csv  2026-10-20 18:01:48.617000  86ea605c7c8b46d98aa72fc2bfd99e25 baseline  SubmissionStatus.PENDING                             


## Experiment 2: post-processing
Same model, one layer at a time, val F1 after each.

### TTA hflip

In [6]:
val_tta = (val + predict(trainer, dataloaders=dm.loader(hflip(dm.val_ds)))) / 2
test_tta = (test + predict(trainer, dataloaders=dm.loader(hflip(dm.test_ds)))) / 2
val_f1(val_tta, y)

/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

0.3321456015110016

### pHash ≤ 2

In [7]:
is_val = (folds.fold == dm.fold).to_numpy()
hashes = phash(DATA_DIR / "train" / f"{i}.jpg" for i in folds.index)
labels = folds[LABELS].to_numpy()

val_dup = blend_duplicates(val_tta, hashes[folds.index.get_indexer(val.index)], hashes[~is_val], labels[~is_val])
test_dup = blend_duplicates(test_tta, phash(DATA_DIR / "test" / f"{i}.jpg" for i in test.index), hashes, labels)
val_f1(val_dup, y)

0.37365251779556274

### Per-class thresholds

In [8]:
thresholds = best_thresholds(val_dup, y)
print(thresholds.round(2).to_dict())
val_f1(val_dup >= thresholds, y)

{'Cat': 0.35, 'Dog': 0.4, 'Moris': 0.2, 'Motya': 0.05, 'Biatrix': 0.15}


0.5473761558532715

### Hierarchy

In [9]:
val_f1(add_parents(val_dup >= thresholds), y)

0.5467010736465454

### Submit

In [10]:
path = ROOT / "models" / RUN_ID / "submission_post.csv"
add_parents(test_dup >= thresholds).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} post: tta+phash+thresholds+hierarchy"

100%|█████████████████████████████████████████| 930k/930k [00:01<00:00, 507kB/s]
Successfully submitted to Cats and Dogs++

In [11]:
!kaggle competitions submissions -c cats-and-dogs | head -4

fileName             date                        description                                                            status                     publicScore  privateScore  
-------------------  --------------------------  ---------------------------------------------------------------------  -------------------------  -----------  ------------  
submission_post.csv  2026-10-20 18:02:59.793000  86ea605c7c8b46d98aa72fc2bfd99e25 post: tta+phash+thresholds+hierarchy  SubmissionStatus.PENDING                              
